In [ ]:
### **1. Import Libraries**

# Import Libraries
import keras # Import the Keras library, a high-level neural networks API
import tensorflow.keras # Import TensorFlow's Keras implementation
from keras.datasets import mnist # Import the MNIST dataset from Keras
from keras.models import Sequential # Import Sequential model, a linear stack of layers
from keras.layers import Dense, Flatten, Dropout # Import core layers: Dense (fully connected), Flatten (for 2D to 1D), Dropout (for regularization)
from keras.layers import Conv2D, MaxPooling2D # Import convolutional layers: Conv2D (2D convolution), MaxPooling2D (2D pooling)
import matplotlib.pyplot as plt # Import Matplotlib for plotting
from keras.constraints import maxnorm # Import maxnorm for weight constraints (regularization)
from keras.utils import np_utils
from tensorflow.keras.optimizers import SGD # Import Stochastic Gradient Descent optimizer

"""### **2. Import Dataset**"""

# Load Dataset and split it into train and test
from keras.datasets import cifar10 # Import the CIFAR-10 dataset from Keras
(x_train, y_train), (x_test, y_test) = cifar10.load_data() # Load CIFAR-10 data, splitting into training and testing sets for images (x) and labels (y)

# Let's check few images of this dataset
# plot first few images
for i in range(9): # Loop through the first 9 images
	# define subplot
	plt.subplot(330 + 1 + i) # Create a 3x3 grid of subplots and select the i-th subplot
	# plot raw pixel data
	plt.imshow(x_train[i], cmap=plt.get_cmap('gray')) # Display the i-th training image (x_train[i]) in grayscale
# show the figure
plt.show() # Show the generated plot of images

"""### **3. Preprocess this dataset**

- The image data cannot be fed directly into the model so we need to perform some operations and process the data to make it ready for our neural network.
"""

# convert class vectors to binary class matrices
from tensorflow.keras.utils import to_categorical # Import to_categorical for one-hot encoding labels
num_classes = 10 # Define the number of classes in the dataset (CIFAR-10 has 10 classes)
y_train = to_categorical(y_train, num_classes) # Convert training labels to one-hot encoded format
y_test = to_categorical(y_test, num_classes) # Convert testing labels to one-hot encoded format

# convert from integers to floats
x_train = x_train.astype('float32') # Convert training image pixel values to float32
x_test = x_test.astype('float32') # Convert testing image pixel values to float32

# normalize to range 0-1
x_train /= 255 # Normalize training image pixel values from [0, 255] to [0, 1]
x_test /= 255 # Normalize testing image pixel values from [0, 255] to [0, 1]

# Let's check normalized images
print('x_train shape:', x_train.shape) # Print the shape of the normalized training data
print(x_train.shape[0], 'train samples') # Print the number of training samples
print(x_test.shape[0], 'test samples') # Print the number of testing samples

"""### **4. Model Building**"""

# Building Models
# Adding layers one by one
# Since this is a classification problem -
# activation function for hidden layer - relu
# activation function for output layer - softmax
# Input shape (since coloured and 32x32 pixels) - (32,32,3)
# Output shape - 10
# Drop out layer is added to deactivate some of the neurons for that layer
model = Sequential() # Initialize a Sequential model
model.add(Conv2D(32, (3, 3), input_shape=(32,32,3), activation='relu', padding='same')) # Add a 2D convolutional layer with 32 filters, 3x3 kernel, ReLU activation, 'same' padding, and input shape for CIFAR-10
model.add(Dropout(0.2)) # Add a Dropout layer with a 20% dropout rate for regularization
model.add(Conv2D(32, (3, 3), activation='relu', padding='same')) # Add another 2D convolutional layer with 32 filters, 3x3 kernel, ReLU activation, 'same' padding
model.add(MaxPooling2D(pool_size=(2, 2))) # Add a 2D MaxPooling layer with a 2x2 pool size to reduce spatial dimensions
model.add(Conv2D(64, (3, 3), activation='relu', padding='same')) # Add a 2D convolutional layer with 64 filters
model.add(Dropout(0.2)) # Add another Dropout layer
model.add(Conv2D(64, (3, 3), activation='relu', padding='same')) # Add another 2D convolutional layer with 64 filters
model.add(MaxPooling2D(pool_size=(2, 2))) # Add another 2D MaxPooling layer
model.add(Conv2D(128, (3, 3), activation='relu', padding='same')) # Add a 2D convolutional layer with 128 filters
model.add(Dropout(0.2)) # Add another Dropout layer
model.add(Conv2D(128, (3, 3), activation='relu', padding='same')) # Add another 2D convolutional layer with 128 filters
model.add(MaxPooling2D(pool_size=(2, 2))) # Add another 2D MaxPooling layer
model.add(Flatten()) # Flatten the 3D output of the convolutional layers into a 1D vector
model.add(Dropout(0.2)) # Add a Dropout layer before the first Dense layer
model.add(Dense(1024, activation='relu', kernel_constraint=maxnorm(3))) # Add a Dense (fully connected) layer with 1024 units, ReLU activation, and max norm constraint
model.add(Dropout(0.2)) # Add a Dropout layer
model.add(Dense(512, activation='relu', kernel_constraint=maxnorm(3))) # Add another Dense layer with 512 units, ReLU activation, and max norm constraint
model.add(Dropout(0.2)) # Add a Dropout layer
model.add(Dense(num_classes, activation='softmax')) # Add the output Dense layer with 10 units (for 10 classes) and softmax activation for multi-class classification
print(model.summary()) # Print a summary of the model architecture

# Model Compilation
# This is a multi class classification problem
# thus loss function - categorical cross entropy and metrics as accuracy
# Optimizer being used is SGD (Stochastic Gradient Descent)
# lr stands for learning rate
opt = SGD(learning_rate=0.01, momentum=0.9, decay=0.0002, nesterov=False) # Configure the SGD optimizer with specified learning rate, momentum, decay, and Nesterov momentum
model.compile(loss='categorical_crossentropy', optimizer=opt, metrics=['accuracy']) # Compile the model, specifying categorical crossentropy as the loss function, the SGD optimizer, and accuracy as the metric